Outline:

1. logistic regression function (sigmoid function)
3. decision boundary
4. logistic loss
5. cost function
6. gradient descent
7. overfitting and underfitting
8. regularization



**classification**

* the output variable y can take on only one of a small handful of possible values

* "binary classification": only two possible outputs/classes/categories

* why is linear regression a badd algorithm for classification problems?
because linear regression can predict outside of range [0,1], and it's highly sensitive to extreme data points (outliers) which can distort the "decision boundary"



**logistic regression function**

* predicts the probability that an input belongs to a specific class

* uses "sigmoid function" to convert inputs into probability value between 0 and 1

* how to interpret results of the logistic regression model: "probability" that a class is 1/True/positive (y = 1) given x and parameters w,b



In [1]:
import numpy as np

In [2]:
x_train = np.array([0., 1, 2, 3, 4, 5])
y_train = np.array([0,  0, 0, 1, 1, 1])

def sigmoid(z):
  return 1 / (1 + np.exp(-z))

# f_wb(x) = g(wx + b) ; g is the sigmoid function
def log_reg_model(x, w, b):
  z = np.dot(x, w) + b    # generalizing to multiple features as well
  g = sigmoid(z)
  return g

# arbitrary parameters
w = 0.5
b = -1

g = log_reg_model(x_train, w, b)
print(g.tolist())


[0.2689414213699951, 0.3775406687981454, 0.5, 0.6224593312018546, 0.7310585786300049, 0.8175744761936437]


* why is MSE/squared error cost bad for logistic regression: it has many local mins because of its non-linear nature (sigmoid component), so it makes the gradient descent unstable




**decision boundary**

* the linear surface that separates data pts into distinct classes, typically where the probability = 0.5
* it sets a treshold where the model switches its predictions from one class to another

read C1_W3_Lab03 for examples



**logistic loss**

* loss: difference of a single example and its target
* cost: measure of the losses over the training set
* the loss function (binary cross-entropy (log loss)):


\begin{cases}
-\log\!\big(f_{\mathbf{w},b}(\mathbf{x}^{(i)})\big), & y^{(i)} = 1 \\[6pt]
-\log\!\big(1 - f_{\mathbf{w},b}(\mathbf{x}^{(i)})\big), & y^{(i)} = 0
\end{cases}


*it is derived from probability theory, a concept called "Maximum Likelihood"

**cost function**


$Loss(f_{w,b}(x^i),b)$ = $-y^{(i)} \log\!\big(f_{\mathbf{w},b}(\mathbf{x}^{(i)})\big)
-
\big(1-y^{(i)}\big)
\log\!\big(1 - f_{\mathbf{w},b}(\mathbf{x}^{(i)})$

$J(\vec{w},b) = \frac{1}{m}\sum_{i=1}^{m}[Loss(f_{w,b}(x^i),b)]$

In [3]:
X_train = np.array([[0.5, 1.5], [1,1], [1.5, 0.5], [3, 0.5], [2, 2], [1, 2.5]])  #(m,n)
Y_train = np.array([0, 0, 0, 1, 1, 1])                                           #(m,)

def logistic_cost(x, y, w, b):
  m = x.shape[0]
  cost = 0.0
  for i in range(m):
    z = np.dot(x[i], w) + b
    g = sigmoid(z)
    cost += -y[i] * np.log(g) - (1 - y[i]) * np.log(1 - g)
  cost = cost / m
  return cost

W = np.array([1,1])
B = -3

print(logistic_cost(X_train, Y_train, W, B))

0.36686678640551745


**gradient descent**

In [4]:
# compute the partial derivatives
def p_derivatives(x, y, w, b):
  m, n = x.shape[0], x.shape[1]
  dj_dw = np.zeros(n)
  dj_db = 0.0
  for i in range(m):
    z = np.dot(x[i], w) + b
    g = sigmoid(z)
    error = g - y[i]
    for j in range(n):
      dj_dw[j] += error * x[i, j]
    dj_db += error
  dj_dw = dj_dw / m
  dj_db = dj_db / m
  return dj_dw, dj_db

# gradient descent algorithm
def grad_descent(x, y, w, b, epoch, alpha):
  for i in range(epoch):
    dj_dw, dj_db = p_derivatives(x, y, w, b)
    w = w - alpha * dj_dw
    b = b - alpha * dj_db
  return w, b

w_tmp  = np.zeros_like(X_train[0])
b_tmp  = 0.
alph = 0.1
iters = 10000

w_, b_ = grad_descent(X_train, Y_train, w_tmp, b_tmp, iters, alph)
print("after gradient descent:")
print(f"w: {w_.tolist()} , b: {b_ :.2f}")


after gradient descent:
w: [5.281230291780549, 5.078156075159833] , b: -14.22


**overfitting**:
* when the model fits the data too well (high variance, which means the model reacts too strongly to training data)
* too many features, little data
* performs extremelly well on training data but poorly on test data

**underfitting**
* model performs poorly on both training and test data
* model is too simple
* features are weak/missing
* not enough training
* high bias, which means it makes big strong assumptions


to reduce overfitting: get more training data, use less features ("feature selection"), use "regularization"

to reduce underfitting: use more features (feature engineering), reduce regularization, train for more epochs, scale features properly

**regularization**

* prevent/reduce overfitting by adding a "penalty" term to the loss function for complexity
* shrinks weights


L1 regularization (Lasso Regression)

* adds sum of absolute values of weights
* drives some weights to 0, making it as a feature selection tool

L2 regularization (Ridge Regression)

* adds sum of squared values of weights
* keeps weights small, close to 0
* keeps weights evenly distributed

both use a regularization parameter ($\lambda$) to control the penalty strength

In [5]:
# L2 regularized cost function for linear regression
def reg_cost_linear_reg(x, y, w, b, lambda_=1):
  m, n = x.shape[0], x.shape[1]
  cost = 0.0
  for i in range(m):
    f_wb_i = np.dot(x[i], w) + b
    cost += (f_wb_i - y[i])**2
  cost = cost / (2 * m)
  reg_cost = 0
  for j in range(n):
    reg_cost += w[j]**2
  reg_cost = reg_cost * (lambda_/(2 * m))
  total_cost = cost + reg_cost
  return total_cost

# regularized cost function for logistic regression
def reg_cost_logistic_reg(x, y, w, b, lambda_=1):
  m, n = x.shape[0], x.shape[1]
  cost = 0.0
  for i in range(m):
    z = np.dot(x[i], w) + b
    g = sigmoid(z)
    # g = np.clip(g, 1e-15, 1 - 1e-15) for stability in case g=0 or g=1 since log(0) is undefined
    cost += -y[i] * np.log(g) - (1 - y[i]) * np.log(1 - g)
  cost = cost / m
  reg_cost = 0
  for j in range(n):
    reg_cost += w[j]**2
  # reg_cost = np.sum(w**2)   a faster and cleaner way
  reg_cost = reg_cost * (lambda_/(2 * m))
  total_cost = cost + reg_cost
  return total_cost


XX_tmp = np.random.rand(5,6)
YY_tmp = np.array([0,1,0,1,0])
WW_tmp = np.random.rand(XX_tmp.shape[1]).reshape(-1,)-0.5
BB_tmp = 0.5
lambda_tmp = 0.7
lin_reg_cost = reg_cost_linear_reg(XX_tmp, YY_tmp, WW_tmp, BB_tmp, lambda_tmp)

print(f"regularized cost (linear regression): {lin_reg_cost}")
print(f"non-regularized cost (logistic regression): {logistic_cost(X_train, Y_train, W, B)}")
print(f"regularized cost (logistic regression): {reg_cost_logistic_reg(X_train, Y_train, W, B, lambda_tmp)}")



regularized cost (linear regression): 0.20196446510029378
non-regularized cost (logistic regression): 0.36686678640551745
regularized cost (logistic regression): 0.4835334530721841


In [6]:
# L2 regularized gradient descent for logistic regression (same as linear regression, only difference is the definition of the hypothesis function)
def reg_p_derivatives(x, y, w, b, lambda_):
  m, n = x.shape
  dj_dw = np.zeros(n)
  dj_db = 0.0
  for i in range(m):
    g = sigmoid(np.dot(x[i], w) + b)
    error = g - y[i]
    for j in range(n):    # can be vectorized without using a for loop, dj_dw += error * x[i]
      dj_dw[j]  += error * x[i, j]
    dj_db += error
  dj_dw = dj_dw / m
  dj_db = dj_db / m
  for j in range(n):    # can also be vectorized without using a for loop, dj_dw += (lambda_ / m) * w
    dj_dw[j] += (lambda_ / m) * w[j]
  return dj_dw, dj_db

def reg_grad_desc(x, y, w, b, lambda_, epoch, alpha):
  for i in range(epoch):
    dj_dw, dj_db = reg_p_derivatives(x, y, w, b, lambda_)
    w = w - alpha * dj_dw
    b = b - alpha * dj_db
  return w, b


x_tmp_train = np.random.rand(5,3)
y_tmp_train = np.array([0,1,0,1,0])
w0 = np.random.rand(x_tmp_train.shape[1])
b0 = 0.5
lambda_tmp_ = 0.7
w_tmp_, b_tmp_ = grad_descent(x_tmp_train, y_tmp_train, w0.copy(), b0, 5000, 0.1)
reg_w_tmp_, reg_b_tmp_ = reg_grad_desc(x_tmp_train, y_tmp_train, w0.copy(), b0, lambda_tmp_, 5000, 0.1)
print("after normal gradient descent:")
print(f"w: {w_tmp_.tolist()}")
print(f"b: {b_tmp_:.2f}")
print("after regularized gradient descent:")
print(f"w: {reg_w_tmp_.tolist()}")
print(f"b: {reg_b_tmp_:.2f}")


after normal gradient descent:
w: [5.834099665727093, -0.6196572257541069, -0.2890346769486422]
b: -3.83
after regularized gradient descent:
w: [0.6825724455956295, 0.03446170765507656, 0.17218872211695307]
b: -0.88
